# SciERC NER — Experiments

| # | Model | Backbone | Head |
|---|-------|----------|------|
| 1 | `bert_linear` | bert-base-cased | Linear |
| 2 | `scibert_linear` | allenai/scibert_scivocab_cased | Linear |
| 3 | `scibert_mlp` | SciBERT | MLP (768→256→13, GELU) |
| 4 | `scibert_crf` | SciBERT | Linear + CRF |
| 5 | `scibert_mlp_crf` | SciBERT | MLP + CRF |
| 6 | `deberta_qlora` | microsoft/deberta-v3-large | QLoRA (4-bit, r=16) + Linear |

Результаты → `results/`, веса → `checkpoints/`.

---
## 0. Setup

In [ ]:
!pip install -q transformers torch seqeval peft bitsandbytes pytorch-crf

In [ ]:
!git clone https://github.com/<username>/scierc-ner.git
%cd scierc-ner

---
## Experiment 1 — BERT-base-cased + Linear

In [ ]:
!python kaggle_run.py --models bert_linear

---
## Experiment 2 — SciBERT + Linear

In [ ]:
!python kaggle_run.py --models scibert_linear

---
## Experiment 3 — SciBERT + MLP

In [ ]:
!python kaggle_run.py --models scibert_mlp

---
## Experiment 4 — SciBERT + CRF

In [ ]:
!python kaggle_run.py --models scibert_crf

---
## Experiment 5 — SciBERT + MLP + CRF

In [ ]:
!python kaggle_run.py --models scibert_mlp_crf

---
## Experiment 6 — DeBERTa-v3-large + QLoRA

In [ ]:
!python kaggle_run.py --models deberta_qlora

---
## Results & Visualisation

In [ ]:
import json
import pandas as pd
from pathlib import Path

RESULTS_DIR  = Path("results")
ENTITY_TYPES = ["Generic", "Material", "Method", "Metric", "OtherScientificTerm", "Task"]
MODEL_ORDER  = ["bert_linear", "scibert_linear", "scibert_mlp",
                "scibert_crf", "scibert_mlp_crf", "deberta_qlora"]
LABELS = {
    "bert_linear":     "BERT\nLinear",
    "scibert_linear":  "SciBERT\nLinear",
    "scibert_mlp":     "SciBERT\nMLP",
    "scibert_crf":     "SciBERT\nCRF",
    "scibert_mlp_crf": "SciBERT\nMLP+CRF",
    "deberta_qlora":   "DeBERTa\nQLoRA",
}

# Load registry and pick latest run per model
registry_path = RESULTS_DIR / "registry.csv"
assert registry_path.exists(), "registry.csv not found — run experiments first"

registry = pd.read_csv(registry_path)
latest = (
    registry.sort_values("run_id")
            .groupby("model_name", as_index=False)
            .last()
)

# Load full results (with history) for each model
results = {}
for _, row in latest.iterrows():
    p = RESULTS_DIR / row["results_file"]
    if p.exists():
        with open(p) as f:
            results[row["model_name"]] = json.load(f)

models = [m for m in MODEL_ORDER if m in results]
print(f"Loaded: {models}\n")

# Summary table
hdr = f"{'Model':<22}  {'Run':^14}  {'MacroF1':>8}" + "".join(f"  {e[:8]:>8}" for e in ENTITY_TYPES)
print(hdr + "\n" + "-" * len(hdr))
for name in models:
    r = results[name]
    m = r["test_metrics"]
    row = f"{name:<22}  {r['run_id']:^14}  {m['macro_f1']:>8.4f}"
    row += "".join(f"  {m.get(e+'_f1', 0):>8.4f}" for e in ENTITY_TYPES)
    print(row)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

PALETTE = plt.cm.tab10(np.linspace(0, 0.6, 6))
colors  = [PALETTE[i] for i in range(len(models))]

macro_f1s = [results[m]["test_metrics"]["macro_f1"] for m in models]
x_labels  = [LABELS.get(m, m) for m in models]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(x_labels, macro_f1s, color=colors, edgecolor="white", linewidth=0.8, zorder=3)
for bar, val in zip(bars, macro_f1s):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_ylabel("Macro F1 (test)", fontsize=11)
ax.set_title("SciERC NER — Macro F1 by Model", fontsize=13, fontweight="bold")
ax.set_ylim(0, min(1.0, max(macro_f1s) * 1.15 + 0.02))
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "macro_f1_comparison.png", dpi=150)
plt.show()

In [ ]:
n_models   = len(models)
n_entities = len(ENTITY_TYPES)
x     = np.arange(n_entities)
width = 0.75 / n_models

fig, ax = plt.subplots(figsize=(13, 5))
for i, model in enumerate(models):
    vals   = [results[model]["test_metrics"].get(e + "_f1", 0) for e in ENTITY_TYPES]
    offset = (i - n_models / 2 + 0.5) * width
    ax.bar(x + offset, vals, width=width * 0.9,
           label=LABELS.get(model, model).replace("\n", " "),
           color=colors[i], edgecolor="white", linewidth=0.4, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(ENTITY_TYPES, fontsize=10)
ax.set_ylabel("F1 (test)", fontsize=11)
ax.set_title("SciERC NER — Per-Entity F1 by Model", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "entity_f1_comparison.png", dpi=150)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_path = RESULTS_DIR / "history.csv"
assert history_path.exists(), "history.csv not found"

history_df = pd.read_csv(history_path)

# Latest run per model
latest_runs = (
    history_df.sort_values("run_id")
              .groupby("model_name")["run_id"]
              .last()
              .to_dict()
)
history_df = history_df[
    history_df.apply(lambda r: latest_runs.get(r["model_name"]) == r["run_id"], axis=1)
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, model in enumerate(models):
    df = history_df[history_df["model_name"] == model].sort_values("epoch")
    if df.empty:
        continue
    label = LABELS.get(model, model).replace("\n", " ")
    axes[0].plot(df["epoch"], df["dev_macro_f1"], marker="o", markersize=4,
                 linewidth=1.8, label=label, color=colors[i])
    axes[1].plot(df["epoch"], df["train_loss"],   marker="o", markersize=4,
                 linewidth=1.8, label=label, color=colors[i])

axes[0].set_title("Dev Macro F1",  fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Macro F1")
axes[0].legend(loc="lower right", fontsize=8)
axes[0].grid(linestyle="--", alpha=0.4)
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].set_title("Train Loss", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].grid(linestyle="--", alpha=0.4)
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle("SciERC NER — Learning Curves", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
